In [390]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn

In [391]:
dataset=pd.read_csv('weather.csv')
dataset

,aes,wmo,rep_date,temp,td,rh,ws,wg,wdir,pres,...,sog,ffmc,dmc,dc,bui,isi,fwi,dsr,opts,calcstatus
0,1012475,71031,2020-01-01 12:00:00,9.5,8.6,94.0,8.6,23.2,261,1007.7,...,0.0,8.9,0.1,85.6,0.1,0.0,0.0,0.0,IDW=R:M=1:,1
1,1012710,71798,2020-01-01 12:00:00,10.2,6.2,76.0,35.8,45.7,252,1007.8,...,0.0,35.6,0.3,1.5,0.4,0.1,0.0,0.0,M=1:,1
2,1014820,71774,2020-01-01 12:00:00,9.4,5.5,76.9,23.4,NaN,255,1006.6,...,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,IDW=RH:WSD:R:M=1:,-3
3,1015630,71927,2020-01-01 12:00:00,11.6,4.4,61.0,12.0,24.1,257,1007.0,...,0.0,36.9,0.5,1.8,0.6,0.0,0.0,0.0,M=1:,1
4,1016640,71778,2020-01-01 12:00:00,9.2,6.9,85.0,45.0,54.0,259,1008.8,...,0.0,28.8,0.2,1.4,0.2,0.0,0.0,0.0,IDW=R:M=1:,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3162100,8502592,71339,2025-01-21 12:00:00,-13.6,-18.9,65.0,18.4,NaN,280,1004.6,...,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,M=1:,-3
3162101,8502799,71665,2025-01-21 12:00:00,-23.0,-28.9,59.0,25.8,NaN,252,996.5,...,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,IDW=WSD:R:M=1:,-3
3162102,8502801,71902,2025-01-21 12:00:00,-23.1,-28.8,61.0,25.8,NaN,252,997.0,...,19.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,IDW=WSD:R:M=1:,-3
3162103,8504175,71825,2025-01-21 12:00:00,-30.9,-36.5,59.0,20.5,NaN,271,1015.4,...,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,M=1:,-3


### **Only including columns that are features and the target output.**

In [392]:
dataset=dataset[['temp','rh','ws','precip', 'rndays','fwi', 'calcstatus']]


### As per our analysis in the previous notebook, I will now be handling missing/incorrect values

## **Filling/Correcting values**

### **Precipitation**

In [393]:
dataset['precip'].isna().value_counts()

precip
False    3162025
True          80
Name: count, dtype: int64

In [394]:
dataset.fillna({'precip':0}, inplace=True)
dataset['precip'].isna().value_counts()

/var/folders/0g/0rngkrh11rz7p3c55850pvjc0000gn/T/ipykernel_14143/3060888321.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dataset.fillna({'precip':0}, inplace=True)


precip
False    3162105
Name: count, dtype: int64

#### Dropped 80 rows where precipitation was Nan/missing.

### **Relative humidity**

In [395]:
(dataset['rh']<0).value_counts()

rh
False    3162087
True          18
Name: count, dtype: int64

In [396]:
dataset.drop(dataset[dataset['rh']<0].index, inplace=True)
(dataset['rh']<0).value_counts()


/var/folders/0g/0rngkrh11rz7p3c55850pvjc0000gn/T/ipykernel_14143/116990623.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dataset.drop(dataset[dataset['rh']<0].index, inplace=True)


rh
False    3162087
Name: count, dtype: int64

#### Dropped 18 rows where relative humidity is less than 0. 

### **Wind speed**

In [397]:
(dataset['ws']<0).value_counts()

ws
False    3162087
Name: count, dtype: int64

## **Target construction**

### Creating a new column that will be the risk band our model will be prediciting. This will be using the calcstatus. 
### If calcstatus=0/-3, risk=LOW because fwi will be missing/cannot be calculated
### if calcstatus=1, risk=MODERATE/HIGH/VERY HIGH/EXTREME. Based on percentile calculation using fwi 

In [398]:
dataset.sort_values(by='fwi', ascending=True, inplace=True)

/var/folders/0g/0rngkrh11rz7p3c55850pvjc0000gn/T/ipykernel_14143/3967847773.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dataset.sort_values(by='fwi', ascending=True, inplace=True)


In [399]:
dataset

,temp,rh,ws,precip,rndays,fwi,calcstatus
0,9.5,94.0,8.6,20.52,0,0.0,1
1246675,1.0,86.7,11.1,1.71,0,0.0,1
1246680,-2.0,74.4,0.0,0.00,1,0.0,1
410816,13.2,97.0,3.7,16.80,0,0.0,1
1246682,2.0,93.2,13.0,3.80,0,0.0,1
...,...,...,...,...,...,...,...
3162100,-13.6,65.0,18.4,1.90,0,NaN,-3
3162101,-23.0,59.0,25.8,3.37,0,NaN,-3
3162102,-23.1,61.0,25.8,3.37,0,NaN,-3
3162103,-30.9,59.0,20.5,0.00,15,NaN,-3


In [400]:
df_calc1=dataset[dataset['calcstatus']==1].copy()
df_calc1_fwi0=df_calc1[df_calc1['fwi']==0].copy()
df_calc1_fwi_not0=df_calc1[df_calc1['fwi']!=0].copy()
df_calc1_fwi0['risk_band']='moderate'
risk_labels=['moderate', 'high', 'very_high', 'extreme']
df_calc1_fwi_not0['risk_band']=pd.qcut(df_calc1_fwi_not0['fwi'], q=[0, 0.5, 0.75, 0.95, 0.99], labels=risk_labels)
combined_df=pd.concat([df_calc1_fwi0, df_calc1_fwi_not0])


In [401]:
dataset=dataset.join(combined_df['risk_band'], how='left')

In [402]:
dataset['risk_band']=dataset['risk_band'].astype('string')
dataset['risk_band']=dataset['risk_band'].fillna('no_risk')

In [403]:
dataset

,temp,rh,ws,precip,rndays,fwi,calcstatus,risk_band
0,9.5,94.0,8.6,20.52,0,0.0,1,moderate
1246675,1.0,86.7,11.1,1.71,0,0.0,1,moderate
1246680,-2.0,74.4,0.0,0.00,1,0.0,1,moderate
410816,13.2,97.0,3.7,16.80,0,0.0,1,moderate
1246682,2.0,93.2,13.0,3.80,0,0.0,1,moderate
...,...,...,...,...,...,...,...,...
3162100,-13.6,65.0,18.4,1.90,0,NaN,-3,no_risk
3162101,-23.0,59.0,25.8,3.37,0,NaN,-3,no_risk
3162102,-23.1,61.0,25.8,3.37,0,NaN,-3,no_risk
3162103,-30.9,59.0,20.5,0.00,15,NaN,-3,no_risk


In [404]:
dataset['risk_band'].isna().sum()

np.int64(0)

In [405]:
dataset['risk_band'].value_counts()

risk_band
moderate     1174738
no_risk      1092504
high          458165
very_high     363510
extreme        73170
Name: count, dtype: Int64

In [406]:
dataset.shape[0]

3162087

In [408]:
dataset.to_csv("df_processed.csv", index=False)